## Marketing Analytics Chapter 3

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Read in the chapter data from GitHub Repo

In [ ]:
bank1 = pd.read_csv("https://raw.githubusercontent.com/librowndev/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter03/Datasets/Bank_Personal_Loan_Modelling-1.csv")

In [ ]:
bank1.head()

In [ ]:
bank2 = pd.read_csv('https://raw.githubusercontent.com/PacktPublishing/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter03/Datasets/Bank_Personal_Loan_Modelling-2.csv')

In [ ]:
bank2.head()

In [ ]:
clothing_customers = pd.read_csv('https://raw.githubusercontent.com/PacktPublishing/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter03/Datasets/Clothing_Customers.csv')

In [ ]:
clothing_customers.head()

In [ ]:
mail_customers = pd.read_csv('https://raw.githubusercontent.com/PacktPublishing/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter03/Datasets/Mall_Customers.csv')

In [ ]:
mail_customers.head()

In [ ]:
age_education = pd.read_csv('https://raw.githubusercontent.com/PacktPublishing/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter03/Datasets/age_education.csv')

In [ ]:
age_education.head()

In [ ]:
customer_offers = pd.read_csv('https://raw.githubusercontent.com/PacktPublishing/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter04/Datasets/customer_offers.csv')

In [ ]:
customer_offers.head()

#### Exercise 3.01

In [ ]:
# 3 (1 -> read datam 2 -> head())
mail_customers.info()


In [ ]:
# 4 rename columns

mail_customers.rename({'Annual Income (k$)': 'Income', "Spending Score (1-100)": 'Spending_score'}, axis=1, inplace=True)

mail_customers.columns

In [ ]:
mail_customers.describe()

## Mall Customers Dataset

**Source:** [Packt – Data Science for Marketing Analytics, 2nd Ed. (Chapter 3)](https://raw.githubusercontent.com/PacktPublishing/Data-Science-for-Marketing-Analytics-Second-Edition/refs/heads/master/Chapter03/Datasets/Mall_Customers.csv)

**Overview:** A small, clean customer-segmentation dataset with **200 rows × 5 columns**, one row per mall customer. A standard teaching set for clustering (e.g., K-means).

| Column | Type | Notes |
|---|---|---|
| `CustomerID` | int | Unique key, 1–200 |
| `Gender` | categorical | Male / Female (~112 Female, ~88 Male) |
| `Age` | int | Range 18–70, mean ≈ 39 |
| `Annual Income (k$)` | int | Range 15–137, mean ≈ 60.6 |
| `Spending Score (1-100)` | int | Mall-assigned score based on behavior and spending; range 1–99, mean ≈ 50 |

### Key Observations
- **No missing values** or malformed rows.
- **Income** is right-skewed, with a few high earners at 120–137k.
- **Spending score** isn't driven by income alone. Income vs. spending score shows roughly **5 groups**:
  - low income / low spending
  - low income / high spending
  - mid income / mid spending (largest group)
  - high income / low spending
  - high income / high spending
- **Age:** younger customers (under ~40) dominate the high-spending groups; older customers cluster at low-to-mid scores.
- **Gender** differences are modest.

### Practical Notes
- Rename `Annual Income (k$)` and `Spending Score (1-100)` to clean names (e.g., `annual_income`, `spending_score`) to avoid issues with spaces and symbols.
- For clustering, **scale** Age, Income, and Spending Score first; use `CustomerID` only as an identifier.


### Exercise 3.02 Traditional Segmentation of Mall Customers

In [ ]:
mail_customers.Income.plot.hist(color='gray')
plt.xlabel('Income')
plt.show()

In [ ]:
mail_customers['Cluster'] = np.where(mail_customers.Income >= 90, 'High earners', np.where(mail_customers.Income < 50, 'Low income', 'Moderate income' ) )

In [ ]:
mail_customers.groupby('Cluster')['Income'].describe()

### Exercise 3.03 Standardizing the Customer Data

## StandardScaler (scikit-learn)

**Module:** `sklearn.preprocessing.StandardScaler`

**Purpose:** Standardizes features by removing the mean and scaling to unit variance, so each feature ends up with **mean = 0** and **standard deviation = 1**.

$$z = \frac{x - \mu}{\sigma}$$

where $\mu$ is the feature's mean and $\sigma$ is its standard deviation, both computed on the training data.

### Basic Usage

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # learn μ, σ from train, then transform
X_test_scaled = scaler.transform(X_test)        # reuse train μ, σ (never refit on test)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [ ]:
cols_to_scale = ['Age', 'Income', 'Spending_score']

mail_scaled = mail_customers.copy(deep=True)


In [ ]:
mail_scaled[cols_to_scale] = scaler.fit_transform(mail_scaled[cols_to_scale])
mail_scaled.describe()

### Exercise 3.04 Calculating Distance Between Clusters

In [ ]:
sel_cols = ['Income', 'Spending_score']
cust3 = mail_scaled[sel_cols].head(3)
cust3

In [ ]:
from scipy.spatial.distance import cdist

cdist(cust3, cust3, metric='euclidean')

#### Exercise 3.05 K-Means Clustering on Mall (Mail) Customers

In [ ]:
from sklearn.cluster import KMeans

#model =KMeans(n_clusters=N, random_state=random_state)

In [ ]:
cluster_cols = ['Income', 'Spending_score']
mail_scaled[cluster_cols].head(3)

In [ ]:
mail_scaled.plot.scatter(x='Income', y='Spending_score', color='blue')
plt.show()

In [ ]:
model = KMeans(n_clusters=5, random_state=42)

model.fit(mail_scaled[cluster_cols])

mail_scaled['Cluster'] = model.predict(mail_scaled[cluster_cols])
mail_scaled.head()


In [ ]:
markers = ['x', '*', '+', '.', '|', '_']

In [ ]:
markers = ['x', '*', '+', '.', '|', '_']

for clust in range(5):
    temp = mail_scaled[mail_scaled.Cluster == clust]
    plt.scatter(temp.Income, temp.Spending_score, marker=markers[clust], cmap='plasma')
    
plt.xlabel('Income')
plt.ylabel('Spending Score')
plt.legend()
plt.show()

In [ ]:
mail_customers['Cluster'] = mail_scaled.Cluster

mail_customers.groupby('Cluster')[['Income', 'Spending_score']].mean().plot.bar(color=['blue', 'red'])
plt.show()

### Activity 3.01 Bank Customer Segmentation

In [ ]:
bank1.head()

In [ ]:
bank1.info()

In [ ]:
# standard scale the income and ccaverage columns
scale_cols = ['Income', 'CCAvg']

bank1_scaled = bank1.copy(deep=True)

bank1_scaled[scale_cols] = scaler.fit_transform(bank1_scaled[scale_cols])
#mail_scaled[cols_to_scale] = scaler.fit_transform(mail_scaled[cols_to_scale])

bank1_scaled.describe()

In [ ]:
bank1_scaled.plot.scatter(x='Income', y='CCAvg', color='blue')

In [ ]:
bank1_scaled[scale_cols].head()


In [ ]:
model1 = KMeans(n_clusters=3, random_state=42)

model1.fit(bank1_scaled[scale_cols])

bank1_scaled['Cluster'] = model1.predict(bank1_scaled[scale_cols])
bank1_scaled.head()

#mail_scaled['Cluster'] = model.predict(mail_scaled[cluster_cols])


In [ ]:

markers = ['x', '*', '+']

for clust in range(3):
    temp = bank1_scaled[bank1_scaled.Cluster == clust]
    plt.scatter(temp.Income, temp.CCAvg, marker=markers[clust], cmap='plasma')
    
plt.xlabel('Income')
plt.ylabel('CCAvg')
plt.legend()
plt.show()

In [ ]:
bank1_scaled.groupby('Cluster')[['Income', 'CCAvg']].mean().plot.bar(color=['blue', 'red'])
plt.show()

In [ ]:
bank1_scaled.groupby('Cluster')[['Age', 'Mortgage','Family','CreditCard', 'Online','Personal Loan']].mean().plot.bar(color=['green', 'blue', 'red', 'orange', 'maroon'])
plt.show()

### Exercise 3.06 High-Dimensional Data

In [ ]:
mall_cols2 = ['Age', 'Income', 'Spending_score']
mail_scaled[mall_cols2].head()


In [ ]:
model2 = KMeans(n_clusters=4, random_state=42)
model2.fit(mail_scaled[mall_cols2])

mail_scaled['Cluster1'] = model2.predict(mail_scaled[mall_cols2])

In [ ]:
from sklearn import decomposition

pca = decomposition.PCA(n_components=2)
pca_res = pca.fit_transform(mail_scaled[mall_cols2])

mail_scaled['pc1'] = pca_res[:,0]
mail_scaled['pc2'] = pca_res[:, 1]

In [ ]:
markers1 = ['x', '*', 'o']

for clust in range(3):
    temp = mail_scaled[mail_scaled.Cluster1 == clust]
    plt.scatter(temp.pc1, temp.pc2, marker=markers1[clust], color='maroon', label='Cluster ' + str(clust))
    
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()
